# Counterfit Drone Detection Assessment - Solution

This notebook configures a binary drone detection image classifier as a real Microsoft Counterfit target, runs three adversarial attacks, and summarizes operational impact using false-negative focused metrics.

The classroom model uses CIFAR-10 airplane images as a lightweight proxy for drone imagery.

In [ ]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
sys.path.append(str(ROOT))

from drone_counterfit_utils import (
    DRONE_CLASSES,
    evaluate_detection_metrics,
    load_npz_dataset,
    require_counterfit,
    run_counterfit_attack_plan,
    write_results_csv,
)
from targets.drone_detection_target import DroneDetectionTarget

attack_plan = json.loads((ROOT / 'configs' / 'counterfit_attack_plan.json').read_text())
target_config = json.loads((ROOT / 'configs' / 'counterfit_target_config.json').read_text())
target_config

## 1. Load the Counterfit target and verify baseline behavior

In [ ]:
counterfit, Counterfit = require_counterfit()

target = DroneDetectionTarget()
target.load()

val_images, val_labels = load_npz_dataset(ROOT / target.data_path)
baseline = evaluate_detection_metrics(target.model, val_images, val_labels, device=target.device)

print(f"Validation images: {len(val_labels)}")
print(f"Baseline accuracy: {baseline['accuracy']:.3f}")
print(f"Drone recall: {baseline['drone_recall']:.3f}")
print(f"Baseline false-negative rate: {baseline['false_negative_rate']:.3f}")
print(f"Mean confidence: {baseline['mean_confidence']:.3f}")

## 2. Run three Counterfit attacks

The attack plan targets the first three validation images, which are drone-proxy samples. This makes false negatives easier to interpret operationally.

In [ ]:
attack_rows = run_counterfit_attack_plan(target, attack_plan)
attack_rows

## 3. Save quantitative results

In [ ]:
results_path = write_results_csv(attack_rows, ROOT / 'results' / 'solution_counterfit_drone_results.csv')
print(results_path)

for row in attack_rows:
    print(
        row['attack'],
        'success=', row['attack_success_rate'],
        'adv_acc=', row['adversarial_accuracy'],
        'fnr=', row['drone_false_negative_rate'],
        'risk=', row['operational_risk'],
    )

## 4. Visualize prediction changes

In [ ]:
sample_indexes = attack_plan['sample_index']
if not isinstance(sample_indexes, list):
    sample_indexes = [sample_indexes]
fig, axes = plt.subplots(len(attack_rows), len(sample_indexes), figsize=(6, 6))
axes = np.array(axes).reshape(len(attack_rows), len(sample_indexes))
for row_idx, row in enumerate(attack_rows):
    final_labels = row['final_labels'].split('|') if row['final_labels'] else []
    for col_idx, sample_idx in enumerate(sample_indexes):
        ax = axes[row_idx][col_idx]
        ax.imshow(val_images[sample_idx].transpose(1, 2, 0))
        predicted = final_labels[col_idx] if col_idx < len(final_labels) else 'unknown'
        ax.set_title(f"{row['attack']}\nfinal={predicted}", fontsize=8)
        ax.axis('off')

plt.tight_layout()
figure_path = ROOT / 'results' / 'solution_prediction_changes.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(figure_path, dpi=160)
figure_path

## 5. Assessment summary

Use the saved CSV and Counterfit run logs to complete the short report in `docs/assessment_report_template.md`. Prioritize attacks that convert true `drone` samples into `no_drone` predictions because those represent missed alerts in the scenario.